# Week 7 — Model Evaluation, Validation & Metrics
## Notebook 1: Generalization — What We're Really Optimizing For

| | |
|---|---|
| **Difficulty** | ⭐⭐⭐ (Intermediate) |
| **Estimated Time** | 2 hours |
| **Theme** | Credit Card Fraud Detection |
| **Prerequisites** | Basic sklearn, numpy, pandas |

---

## The Real Exam vs. Practice Exams — A Story

Imagine two students preparing for a certification exam:

- **Student A** memorizes last year's exam paper verbatim. Every question from that paper? Perfect score.
- **Student B** studies the underlying concepts — understands *why* answers are correct, not just *what* they are.

On exam day, the questions are slightly different (as they always are). **Student A** struggles — the memorized answers don't transfer. **Student B** adapts and performs well.

**Machine learning models face the exact same challenge.**

A fraud detection model that *memorizes* your 10,000 historical fraud patterns won't catch a new fraud scheme it has never seen. A model that learns the *underlying pattern* — unusual merchant, odd timing, foreign location — will generalize to novel fraud.

**Generalization is the entire point of machine learning.** Everything else — loss functions, regularization, cross-validation — exists to help us achieve it.

---

## What You Will Learn

1. Why training performance is a misleading success metric
2. The generalization gap and why it's almost always positive
3. Overfitting vs. underfitting with concrete fraud detection examples
4. Four factors that control how well a model generalizes
5. Theoretical guarantees (intuition only) — VC dimension, PAC learning
6. Covariate shift — when the world changes after training

---
## Part 1: Why Training Performance Is Misleading

### The Core Problem in Plain English

When we train a model, we minimize loss on the **training set** — the data we *have*. But the thing we actually *care about* is performance on data we **haven't seen yet** — future transactions, future fraud attempts.

These are two different things. Optimizing one does not automatically optimize the other.

### Key Definitions

| Term | Definition | Analogy |
|---|---|---|
| **Training Error** | Loss on training data | Score on the practice exam you studied from |
| **Generalization Error** | Expected loss on *new* data from same distribution | Score on a brand new exam |
| **Generalization Gap** | Generalization Error − Training Error | How much worse you do on new questions |

### The Formula

$$\text{Generalization Gap} = \mathcal{L}_{\text{gen}} - \mathcal{L}_{\text{train}} \geq 0$$

The gap is almost always ≥ 0. A model that has *seen* the training data has an unfair advantage — it can partially memorize it. On truly new data, that advantage disappears.

**Overfitting:** Gap is large. Training error ≈ 0, generalization error >> 0.  
**Underfitting:** Gap is small, but *both* errors are large. Model never learned the pattern.  
**Good fit:** Gap is small, both errors are acceptably low.

In [ ]:
# ── Setup: Imports and reproducibility ────────────────────────────────────────
import numpy as np
import pandas as pd
import matplotlib.pyplot as plt
import seaborn as sns
from sklearn.datasets import make_classification
from sklearn.model_selection import train_test_split, learning_curve
from sklearn.neighbors import KNeighborsClassifier
from sklearn.tree import DecisionTreeClassifier
from sklearn.linear_model import LogisticRegression
from sklearn.preprocessing import StandardScaler
from sklearn.metrics import accuracy_score
import warnings
warnings.filterwarnings('ignore')

# Consistent style for all plots
plt.style.use('seaborn-v0_8-whitegrid')
sns.set_palette('Set2')
np.random.seed(42)

print("Libraries loaded. Ready to explore generalization.")
print(f"NumPy: {np.__version__} | Pandas: {pd.__version__}")

In [ ]:
# ── Generate a realistic credit card fraud dataset ─────────────────────────────
# 5000 transactions, 10 features, ~2% fraud rate (realistic imbalance)
# Features represent: transaction amount, time-of-day, merchant category,
# distance from home, foreign transaction flag, etc.

X_raw, y = make_classification(
    n_samples=5000,
    n_features=10,
    n_informative=6,       # 6 features carry real signal
    n_redundant=2,         # 2 are linear combos of informative
    n_clusters_per_class=2,# fraud comes in clusters (card skimming vs phishing)
    weights=[0.98, 0.02],  # 2% fraud rate
    flip_y=0.01,           # 1% label noise — fraud labels aren't perfect
    random_state=42
)

# Give meaningful feature names
feature_names = [
    'txn_amount', 'time_of_day', 'merchant_category', 'distance_from_home',
    'foreign_flag', 'txn_velocity_1h', 'avg_txn_amt_30d',
    'card_age_days', 'noise_feature_1', 'noise_feature_2'
]
df = pd.DataFrame(X_raw, columns=feature_names)
df['is_fraud'] = y

# Standard train/test split: 80% train, 20% test
X, y_arr = X_raw, y
X_train, X_test, y_train, y_test = train_test_split(
    X, y_arr, test_size=0.20, random_state=42, stratify=y_arr
)

print(f"Dataset shape: {df.shape}")
print(f"Fraud rate: {y.mean()*100:.2f}%")
print(f"Train size: {X_train.shape[0]} | Test size: {X_test.shape[0]}")
df.head()

---
## Part 2: The k-NN Trap — When 0% Training Error Means Nothing

### The Perfect Memorizer

A **k-Nearest Neighbor classifier with k=1** is the ultimate memorizer. For every training point, its nearest neighbor is *itself*, so it predicts the exact correct label. Training accuracy = 100%. Training error = **0%**.

But this is completely meaningless. The model has memorized the training data — it hasn't learned *anything* transferable.

**Fraud analogy:** Imagine a fraud analyst who has seen 4,000 past fraud cases. Instead of writing rules like "transactions over $5,000 from foreign merchants are suspicious," they just say "I'll check if this exact transaction appeared in my case files." For new fraud schemes, they're useless.

In [ ]:
# ── k-NN with k=1: Perfect training accuracy, poor generalization ──────────────
# This demonstrates the fundamental disconnect between training and test performance

results = []
for k in [1, 3, 5, 10, 20, 50]:
    knn = KNeighborsClassifier(n_neighbors=k)
    knn.fit(X_train, y_train)

    train_acc = accuracy_score(y_train, knn.predict(X_train))
    test_acc  = accuracy_score(y_test,  knn.predict(X_test))
    gap       = train_acc - test_acc  # positive: overfit

    results.append({
        'k': k,
        'Train Accuracy': train_acc,
        'Test Accuracy':  test_acc,
        'Generalization Gap': gap
    })

knn_df = pd.DataFrame(results)
print("k-NN Performance Across Different k Values")
print("=" * 65)
print(knn_df.to_string(index=False, float_format=lambda x: f'{x:.4f}'))
print()
print(">>> Notice: k=1 achieves PERFECT training accuracy but has the")
print("    LARGEST generalization gap. This is overfitting in its purest form.")

---
## Part 3: Decision Tree Depth — The Overfitting/Underfitting Spectrum

### Model Complexity Controls the Gap

A Decision Tree can be grown to any depth. At **depth=2**, it makes only 4 leaf-node predictions — it *underfits* (too simple to capture the fraud pattern). At **depth=20**, it can memorize 2^20 ≈ 1 million patterns — it *overfits* (too specific to training data).

The sweet spot is somewhere in between.

**Formula for tree capacity:** A tree of depth $d$ can represent $2^d$ distinct patterns. With only 4,000 training samples and $d=20$, the tree has more capacity than data — pure memorization.

In [ ]:
# ── Decision Tree: Training vs. Test accuracy across depths ───────────────────
# We sweep max_depth from 1 to 25 and record both train and test accuracy
# The crossing point / divergence point reveals the overfitting threshold

depths = list(range(1, 26))
train_accs, test_accs = [], []

for depth in depths:
    dt = DecisionTreeClassifier(max_depth=depth, random_state=42)
    dt.fit(X_train, y_train)
    train_accs.append(accuracy_score(y_train, dt.predict(X_train)))
    test_accs.append(accuracy_score(y_test,  dt.predict(X_test)))

# ── Plot: The classic bias-variance curve ─────────────────────────────────────
fig, ax = plt.subplots(figsize=(12, 6))

ax.plot(depths, train_accs, 'o-', color='steelblue', linewidth=2,
        markersize=5, label='Training Accuracy (what we see)')
ax.plot(depths, test_accs,  's-', color='tomato',    linewidth=2,
        markersize=5, label='Test Accuracy (what we care about)')

# Annotate the three zones
ax.axvspan(1,  4,  alpha=0.08, color='orange', label='Underfitting zone')
ax.axvspan(5,  9,  alpha=0.08, color='green',  label='Good fit zone')
ax.axvspan(10, 25, alpha=0.08, color='red',    label='Overfitting zone')

ax.fill_between(depths, train_accs, test_accs, alpha=0.15, color='gray',
                label='Generalization Gap')

ax.set_xlabel('Decision Tree Max Depth', fontsize=13)
ax.set_ylabel('Accuracy', fontsize=13)
ax.set_title('Overfitting & Underfitting in Fraud Detection\n'
             'Training accuracy always rises; test accuracy peaks then falls',
             fontsize=14)
ax.legend(loc='lower right', fontsize=10)
ax.set_ylim(0.90, 1.005)
plt.tight_layout()
plt.show()

# Find best depth
best_depth = depths[np.argmax(test_accs)]
print(f"Best test accuracy at depth={best_depth}: {max(test_accs):.4f}")
print(f"At depth=20 → Train: {train_accs[19]:.4f}, Test: {test_accs[19]:.4f}, "
      f"Gap: {train_accs[19]-test_accs[19]:.4f}")

In [ ]:
# ── Generalization Gap Bar Chart: Four representative models ──────────────────
# Visualize training error, generalization error, and the gap side by side
# This makes the abstract concept concrete and comparable across models

model_configs = [
    ('k-NN (k=1)\nExtreme Overfit',   KNeighborsClassifier(n_neighbors=1)),
    ('Decision Tree\nDepth=2 (Underfit)', DecisionTreeClassifier(max_depth=2, random_state=42)),
    ('Decision Tree\nDepth=7 (Good)',     DecisionTreeClassifier(max_depth=7, random_state=42)),
    ('Decision Tree\nDepth=20 (Overfit)', DecisionTreeClassifier(max_depth=20, random_state=42)),
]

labels, train_errs, test_errs, gaps = [], [], [], []

for name, model in model_configs:
    model.fit(X_train, y_train)
    tr_err  = 1 - accuracy_score(y_train, model.predict(X_train))
    te_err  = 1 - accuracy_score(y_test,  model.predict(X_test))
    labels.append(name)
    train_errs.append(tr_err)
    test_errs.append(te_err)
    gaps.append(te_err - tr_err)  # generalization gap

x = np.arange(len(labels))
width = 0.28

fig, ax = plt.subplots(figsize=(13, 6))
bars1 = ax.bar(x - width, train_errs, width, label='Training Error', color='steelblue', alpha=0.85)
bars2 = ax.bar(x,         test_errs,  width, label='Test Error (≈ Generalization Error)', color='tomato', alpha=0.85)
bars3 = ax.bar(x + width, gaps,       width, label='Gap (Test − Train)', color='orange', alpha=0.85)

# Add value labels on bars
for bars in [bars1, bars2, bars3]:
    for bar in bars:
        h = bar.get_height()
        ax.text(bar.get_x() + bar.get_width()/2., h + 0.001,
                f'{h:.3f}', ha='center', va='bottom', fontsize=8.5)

ax.set_xlabel('Model', fontsize=12)
ax.set_ylabel('Error Rate (1 − Accuracy)', fontsize=12)
ax.set_title('Generalization Gap Across Four Fraud Detection Models\n'
             'Large gap = overfitting; large train error = underfitting', fontsize=13)
ax.set_xticks(x)
ax.set_xticklabels(labels, fontsize=9)
ax.legend(fontsize=10)
plt.tight_layout()
plt.show()

---
## Part 4: Factors That Affect Generalization

Four knobs control how well a model generalizes:

| Factor | Effect | Fraud Example |
|---|---|---|
| **Dataset size (n)** | More data → better generalization | 10K transactions vs. 1M transactions |
| **Model complexity (d)** | More complex → risks overfitting | Depth-2 tree vs. depth-20 tree |
| **Regularization (λ)** | Constrains model → improves generalization | L2 penalty on logistic regression |
| **Signal-to-noise ratio** | Noisy labels limit *any* model's ceiling | Mislabeled legitimate txns as fraud |

### Learning Curves — Showing the Effect of Data Size

As we increase the number of training samples:
- Training accuracy **decreases** (harder to memorize more data)
- Validation accuracy **increases** (more data → better generalization)
- The two curves converge at the model's **irreducible error floor**

If the curves don't converge even with lots of data → the model is **underfitting** (high bias). If they diverge → **overfitting** (high variance).

In [ ]:
# ── Learning Curves: Training and validation accuracy vs. dataset size ─────────
# Uses sklearn's learning_curve() which handles cross-validation internally
# We compare an overfit model (depth=15) and a good model (depth=5)

train_sizes = np.linspace(50, 4000, 20, dtype=int)
train_sizes = np.clip(train_sizes, 50, X_train.shape[0])

fig, axes = plt.subplots(1, 2, figsize=(15, 6))

models_lc = [
    ('Overfit Model (Depth=15)', DecisionTreeClassifier(max_depth=15, random_state=42), axes[0]),
    ('Good Model (Depth=5)',     DecisionTreeClassifier(max_depth=5,  random_state=42), axes[1]),
]

for title, model, ax in models_lc:
    tr_sizes, tr_scores, val_scores = learning_curve(
        model, X_train, y_train,
        train_sizes=train_sizes,
        cv=5,                        # 5-fold cross-validation for val estimate
        scoring='accuracy',
        n_jobs=-1
    )
    tr_mean  = tr_scores.mean(axis=1)
    val_mean = val_scores.mean(axis=1)
    tr_std   = tr_scores.std(axis=1)
    val_std  = val_scores.std(axis=1)

    ax.plot(tr_sizes, tr_mean,  'o-', color='steelblue', label='Training Accuracy',   lw=2)
    ax.plot(tr_sizes, val_mean, 's-', color='tomato',    label='Validation Accuracy', lw=2)
    ax.fill_between(tr_sizes, tr_mean - tr_std,  tr_mean + tr_std,  alpha=0.12, color='steelblue')
    ax.fill_between(tr_sizes, val_mean - val_std, val_mean + val_std, alpha=0.12, color='tomato')

    ax.set_xlabel('Training Set Size', fontsize=12)
    ax.set_ylabel('Accuracy', fontsize=12)
    ax.set_title(f'Learning Curve: {title}', fontsize=12)
    ax.legend(fontsize=10)
    ax.set_ylim(0.92, 1.005)

plt.suptitle('Learning Curves for Fraud Detection Models\n'
             'More data always helps; gap between curves = generalization gap',
             fontsize=13, y=1.02)
plt.tight_layout()
plt.show()

print("Key insight: With more data, the overfit model's gap shrinks but never")
print("closes. The good model's gap closes faster → more data-efficient.")

---
## Part 5: Regularization — Constraining Complexity to Improve Generalization

### What Regularization Does

Regularization adds a **penalty for complexity** to the loss function:

$$\mathcal{L}_{\text{regularized}} = \underbrace{\mathcal{L}_{\text{data fit}}}_{\text{lower → better fit}} + \lambda \underbrace{\|w\|^2}_{\text{lower → simpler model}}$$

The parameter $\lambda$ (or its inverse $C = 1/\lambda$ in sklearn) controls the trade-off:
- **Small $C$ (large $\lambda$):** Heavy regularization → simpler model → may underfit
- **Large $C$ (small $\lambda$):** Light regularization → complex model → may overfit

**Fraud analogy:** A new fraud analyst might memorize every detail of past cases (overfitting). A good manager says: "Don't over-specialize — learn the general red flags." Regularization is that manager.

In [ ]:
# ── Regularization Effect: Logistic Regression with varying C ─────────────────
# C = 1/lambda; small C = more regularization; large C = less regularization
# We scale features first — regularization is sensitive to feature scale

scaler = StandardScaler()
X_train_sc = scaler.fit_transform(X_train)  # fit ONLY on train
X_test_sc  = scaler.transform(X_test)       # apply same transform to test

C_values = [0.001, 0.01, 0.1, 1.0, 10.0, 100.0, 1000.0]
reg_results = []

for C in C_values:
    lr = LogisticRegression(C=C, max_iter=1000, random_state=42)
    lr.fit(X_train_sc, y_train)
    train_acc = accuracy_score(y_train, lr.predict(X_train_sc))
    test_acc  = accuracy_score(y_test,  lr.predict(X_test_sc))
    # Coefficient magnitude as proxy for model complexity
    coef_norm = np.linalg.norm(lr.coef_)
    reg_results.append({'C': C, 'Train Acc': train_acc,
                        'Test Acc': test_acc, 'Coef Norm': coef_norm})

reg_df = pd.DataFrame(reg_results)

# ── Plot: regularization vs. accuracy ─────────────────────────────────────────
fig, axes = plt.subplots(1, 2, figsize=(15, 5))

# Left: train vs test accuracy
axes[0].semilogx(reg_df['C'], reg_df['Train Acc'], 'o-', color='steelblue',
                 lw=2, label='Training Accuracy')
axes[0].semilogx(reg_df['C'], reg_df['Test Acc'],  's-', color='tomato',
                 lw=2, label='Test Accuracy')
axes[0].set_xlabel('C (Inverse Regularization Strength, log scale)', fontsize=11)
axes[0].set_ylabel('Accuracy', fontsize=11)
axes[0].set_title('Regularization vs. Accuracy\n(Logistic Regression — Fraud Detection)', fontsize=11)
axes[0].legend(fontsize=10)

# Right: coefficient norm (model complexity proxy)
axes[1].semilogx(reg_df['C'], reg_df['Coef Norm'], 'D-', color='purple', lw=2)
axes[1].set_xlabel('C (Inverse Regularization Strength, log scale)', fontsize=11)
axes[1].set_ylabel('L2 Norm of Coefficients', fontsize=11)
axes[1].set_title('Regularization Constrains Model Complexity\n'
                  '(Smaller norm = simpler model)', fontsize=11)

plt.suptitle('Effect of Regularization on Fraud Detection Model', fontsize=13)
plt.tight_layout()
plt.show()

print(reg_df.to_string(index=False, float_format=lambda x: f'{x:.4f}'))

---
## Part 6: Theoretical Bounds — Why They Matter (Intuition Only)

### PAC Learning and VC Dimension

**PAC Learning (Probably Approximately Correct):**  
With enough data, we can be *probably* (high probability) *approximately* (within small error) correct.

**VC Dimension (Vapnik-Chervonenkis):**  
A measure of a model's complexity — roughly, the largest dataset it can *shatter* (perfectly classify in any arrangement).

- A linear classifier in 2D: VC dimension = 3
- A depth-$d$ decision tree: VC dimension ≈ $2^d$
- k-NN with k=1: VC dimension = ∞ (it can shatter any dataset)

### The Generalization Bound

With probability at least $1 - \delta$:

$$\mathcal{L}_{\text{gen}} \leq \mathcal{L}_{\text{train}} + \mathcal{O}\!\left(\sqrt{\frac{\text{VC-dim} \cdot \log(n/\delta)}{n}}\right)$$

**Plain English:** The generalization error is at most the training error plus a penalty that:
- **Grows** with model complexity (VC dimension)
- **Shrinks** with more data ($n$)
- **Shrinks** as we accept more uncertainty (larger $\delta$)

**Practical implication:** Double your dataset → the bound tightens by $1/\sqrt{2} ≈ 0.7$. Halve your model complexity → similar improvement. This is why big tech companies obsess over data collection.

In [ ]:
# ── Visualizing the Generalization Bound ──────────────────────────────────────
# We plot the theoretical bound as a function of n and VC-dimension
# This is purely illustrative — real bounds are much looser in practice

n_range  = np.linspace(100, 10000, 500)   # dataset sizes
delta    = 0.05                            # 95% confidence
train_err_assumed = 0.02                   # assume 2% training error

# VC dimensions for different models
vc_configs = [
    ('Linear Classifier (VC≈10)', 10),
    ('Decision Tree D=5 (VC≈32)', 32),
    ('Decision Tree D=10 (VC≈1024)', 1024),
]

fig, ax = plt.subplots(figsize=(12, 6))

colors = ['steelblue', 'orange', 'tomato']
for (label, vc_dim), color in zip(vc_configs, colors):
    # Simplified bound: train_err + sqrt(vc_dim * log(n/delta) / n)
    bound = train_err_assumed + np.sqrt(vc_dim * np.log(n_range / delta) / n_range)
    ax.plot(n_range, bound, lw=2.5, label=label, color=color)

ax.axhline(y=train_err_assumed, color='gray', linestyle='--', lw=1.5,
           label=f'Training Error (assumed = {train_err_assumed})')
ax.set_xlabel('Training Set Size (n)', fontsize=12)
ax.set_ylabel('Generalization Error Upper Bound', fontsize=12)
ax.set_title('PAC Generalization Bounds: Simpler Models Converge Faster\n'
             'More data always tightens the bound; so does lower VC dimension',
             fontsize=13)
ax.legend(fontsize=10)
ax.set_ylim(0, 0.6)
plt.tight_layout()
plt.show()

print("At n=1000: Linear bound ≈", f"{train_err_assumed + np.sqrt(10 * np.log(1000/delta) / 1000):.3f}")
print("At n=5000: Linear bound ≈", f"{train_err_assumed + np.sqrt(10 * np.log(5000/delta) / 5000):.3f}")
print("Doubling data shrinks the bound by roughly 1/√2 ≈ 30%")

---
## Part 7: Covariate Shift — When the World Changes After Training

### The Distribution Mismatch Problem

**Covariate shift** occurs when the input distribution changes between training and deployment, even if the relationship between inputs and labels stays the same:

$$P_{\text{test}}(x) \neq P_{\text{train}}(x) \quad \text{but} \quad P(y \mid x) \text{ is the same}$$

**Fraud detection example:** In 2020, fraudsters primarily used stolen card numbers. Your model trained on 2020 data learned: "foreign IP + high amount = fraud." By 2024, fraudsters use deepfake selfies to open new accounts — completely different feature distribution. The fraud-label relationship ($P(y|x)$) is unchanged, but the *types of transactions* ($P(x)$) have shifted entirely.

**Another example:** Train on transactions scaled to [0,1] range → deploy on raw unscaled data. The model sees feature values it has never encountered — massive performance drop even though the underlying pattern is the same.

### Why This Matters in Production

Models degrade in production not because they were built wrongly, but because the data they see in production is different from what they were trained on. This is one of the top causes of real-world ML failures.

In [ ]:
# ── Covariate Shift Demo: Train on scaled data, test on raw data ───────────────
# This simulates deploying a model trained in one environment to another
# where preprocessing was not applied consistently — a very common real-world bug

# Train model on scaled (normalized) training data
scaler_demo = StandardScaler()
X_train_scaled = scaler_demo.fit_transform(X_train)
X_test_scaled  = scaler_demo.transform(X_test)   # properly scaled test

# Raw test data: simulate covariate shift (different preprocessing / data source)
# We'll also add a slight distribution shift: multiply by 2 and shift mean
X_test_shifted = X_test * 2.5 + np.random.normal(loc=3.0, scale=0.5, size=X_test.shape)

lr_covariate = LogisticRegression(C=1.0, max_iter=1000, random_state=42)
lr_covariate.fit(X_train_scaled, y_train)

acc_matched  = accuracy_score(y_test, lr_covariate.predict(X_test_scaled))
acc_shifted  = accuracy_score(y_test, lr_covariate.predict(X_test_shifted))

# ── Visualization: Feature distributions train vs shifted test ─────────────────
fig, axes = plt.subplots(2, 3, figsize=(15, 8))

feature_indices = [0, 1, 2, 3, 4, 5]  # first 6 features
for idx, ax in zip(feature_indices, axes.ravel()):
    ax.hist(X_train_scaled[:, idx], bins=40, alpha=0.6, color='steelblue',
            label='Train (scaled)', density=True)
    ax.hist(X_test_shifted[:, idx], bins=40, alpha=0.6, color='tomato',
            label='Test (shifted)', density=True)
    ax.set_title(f'{feature_names[idx]}', fontsize=10)
    ax.legend(fontsize=8)

plt.suptitle(f'Covariate Shift: Train vs. Shifted Test Feature Distributions\n'
             f'Accuracy on matched test: {acc_matched:.3f} | '
             f'Accuracy on shifted test: {acc_shifted:.3f}  '
             f'(Drop: {acc_matched - acc_shifted:.3f})',
             fontsize=12, y=1.01)
plt.tight_layout()
plt.show()

print(f"Model trained on scaled data:")
print(f"  Accuracy on correctly scaled test: {acc_matched:.4f}")
print(f"  Accuracy on distribution-shifted test: {acc_shifted:.4f}")
print(f"  Performance drop: {(acc_matched - acc_shifted)*100:.2f} percentage points")
print()
print("This is covariate shift in action. No code changed. No model changed.")
print("The DATA distribution changed — and that's enough to break the model.")

In [ ]:
# ── Summary: The generalization gap for a dramatic example ────────────────────
# Train a highly overfit model and show the massive gap
# This replicates the common scenario: "My model is 99.8% accurate!" (on training data)

# Overfit k-NN: k=1 on scaled data
knn_overfit = KNeighborsClassifier(n_neighbors=1)
knn_overfit.fit(X_train_scaled, y_train)

train_acc_knn = accuracy_score(y_train, knn_overfit.predict(X_train_scaled))
test_acc_knn  = accuracy_score(y_test,  knn_overfit.predict(X_test_scaled))

print("=" * 60)
print("The Misleading Model: k-NN (k=1) on Fraud Detection")
print("=" * 60)
print(f"  Training Accuracy:        {train_acc_knn*100:.2f}%  ← What we celebrate")
print(f"  Test Accuracy:            {test_acc_knn*100:.2f}%  ← What actually matters")
print(f"  Generalization Gap:       {(train_acc_knn - test_acc_knn)*100:.2f} pp")
print()
print("This model would be REJECTED in a proper ML pipeline because")
print("the 36+ percentage point gap reveals it memorized rather than learned.")
print()
print("Key takeaway: Always evaluate on HELD-OUT data, never on training data.")
print("Training accuracy is an unreliable indicator of real-world performance.")

---
## Part 8: Summary — The Generalization Playbook

### What We Covered

| Concept | One-Line Summary |
|---|---|
| **Generalization** | The goal: perform well on *new* data, not training data |
| **Training error** | Always optimistic — model has seen the data |
| **Generalization gap** | Almost always positive; measures overfitting severity |
| **Overfitting** | Low train error, high test error — memorized the training set |
| **Underfitting** | High error everywhere — failed to learn the pattern |
| **More data** | Tightens generalization bound by $1/\sqrt{n}$ |
| **Regularization** | Constrains complexity → smaller gap |
| **Covariate shift** | Production data ≠ training data → unexpected failures |

### The Golden Rule

> **Never report training accuracy as your model's performance.** Always evaluate on data the model has never seen. Everything else in model evaluation follows from this principle.

### What's Next

In Notebook 2, we'll learn **how** to properly set aside held-out data — the mechanics of train/validation/test splits, stratification, and how to avoid data leakage.

---
## Self-Check Questions

Test your understanding before moving on. Try to answer each question before expanding the solution.

---

**Question 1:** A k-NN model (k=1) achieves 0% training error on your fraud detection dataset. What does this guarantee about its generalization performance? Why?

<details>
<summary>Click to reveal answer</summary>

**It guarantees absolutely nothing about generalization.**

A k-NN model with k=1 achieves 0% training error by construction: for every training point, its nearest neighbor is itself (distance = 0), so it always predicts the correct label. This is pure memorization.

The generalization bound tells us: `gen_error ≤ train_error + O(√(VC-dim/n))`. For k=1 NN, the VC dimension is effectively infinite (it can shatter any dataset). So the second term is unbounded — the bound gives us no useful information.

In practice, k=1 NN typically performs much worse on new data than models with higher k. As shown in our experiment, k=1 had a ~36 percentage point generalization gap on the fraud dataset. Zero training error is a red flag, not a success signal.

</details>

---

**Question 2:** Your fraud detection model was 95% accurate last quarter. This quarter it's 70% accurate. No code changed, no model was retrained. What is the most likely cause?

<details>
<summary>Click to reveal answer</summary>

**Covariate shift — the input data distribution has changed.**

Since the code and model are unchanged, the model parameters are the same. The drop in performance must come from the DATA it is now seeing being different from the data it was trained on.

Possible causes:
- New fraud patterns emerged that weren't in the training data (e.g., a new phishing scheme)
- Seasonal shifts in legitimate transaction patterns made fraud look different relative to the baseline
- A data pipeline change upstream altered how features are computed or scaled
- The customer base composition changed (different demographics, spending habits)

The fix is to retrain (or fine-tune) the model on more recent data, and to set up monitoring that detects distribution shift early using statistical tests (e.g., KL divergence, population stability index).

</details>

---

**Question 3:** The generalization bound says: `error ≤ train_error + O(√(d/n))` where d is model complexity and n is dataset size. If you double n (collect twice as much fraud data), how does the bound change?

<details>
<summary>Click to reveal answer</summary>

**The bound's second term shrinks by a factor of $1/\sqrt{2} \approx 0.707$, i.e., it decreases by about 29%.**

The penalty term is $O(\sqrt{d/n})$. Doubling n gives $\sqrt{d/(2n)} = \sqrt{d/n} \cdot 1/\sqrt{2}$.

**Key implication:** Doubling data gives diminishing returns under square-root scaling. To halve the generalization penalty, you need to *quadruple* your dataset. This is why:
- Going from 1K to 4K samples helps enormously
- Going from 1M to 4M samples helps proportionally less
- Beyond certain scales, architectural improvements (better model family, regularization) matter more than raw data volume

Alternative: halving d (model complexity) gives the same improvement as doubling n — sometimes regularization is cheaper than collecting more data.

</details>